# Embedding mix — 1 tokenizer · nhiều bảng embedding · 1 encoder

**Settings:** Accelerator = **GPU T4 ×2** · Internet = **On**

```
"Bajetigu thu nin …"
      │  tokenizer của ENCODER (1 cái)  → input_ids
      ├──► E_0 : bảng embedding của encoder (cnerg_muril)            ← vẫn được train
      ├──► E_1 : bảng của google/muril-base-cased                    ← đóng băng
      └──► E_2 : bảng của MuRIL sau MLM (checkpoints/mlm/…)           ← đóng băng
                 e = E_0[id] + Σ a_k · (E_k[id] − E_0[id])      a_k khởi tạo = 0
      │
      ▼  + position + LayerNorm → 1 encoder (cnerg_muril, 12 block) → pooling → head
```

- **`embed_mix_mode: token`**: mỗi token có trọng số `a_k` riêng (router `Linear(768→K)` khởi tạo 0).
  **`global`**: một trọng số chung cho mỗi nguồn.
- Nguồn phải **dùng chung vocab** với encoder. MuRIL, cnerg_muril và MuRIL-MLM đều dùng cùng một
  file vocab (197.285 mục). mBERT/XLM-R bị **từ chối**, vì chỉ khớp ~80% token và phần lệch
  chính là các từ Kanglish.
- Log mỗi run in `embed_mix a[nguồn]`: gần **0** nghĩa là nguồn đó không được dùng.

> **Kỳ vọng thực tế.** Đo trước: trên token Kanglish, bảng MuRIL-MLM gần như **trùng** bảng MuRIL
> gốc (cosine ≈ 0,998). MLM thay đổi chủ yếu **encoder**, không phải bảng. Vì vậy notebook chạy
> kèm một **run đối chứng dùng thẳng encoder MLM**. Nếu run đó thắng mà mix không thắng, câu trả
> lời là dùng encoder MLM, không phải trộn bảng.

In [ ]:
import os, subprocess, sys

REPO, BRANCH, WORK = "trong5nhan6/Text", "main", "/kaggle/working"
TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("GH_TOKEN")
except Exception:
    pass
url = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
hide = (lambda s: s.replace(TOKEN, "***")) if TOKEN else (lambda s: s)

os.chdir(WORK)
cmd = (["git", "-C", "repo", "pull", "--ff-only"] if os.path.isdir("repo/.git")
       else ["git", "clone", "--depth", "1", "-b", BRANCH, url, "repo"])
r = subprocess.run(cmd, capture_output=True, text=True)
print(hide((r.stdout + r.stderr).strip()))
if r.returncode:
    raise SystemExit("git that bai -- kiem tra Internet = On, repo Public")

os.chdir(f"{WORK}/repo"); sys.path.insert(0, os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
!pip -q install ftfy sentencepiece gdown
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1) Tải checkpoint MLM

File `mlm_muril.zip` (~1,4 GB) do `pretrain_mlm.ipynb` sinh ra, để trên Google Drive (chế độ
*Anyone with the link*). Bên trong là `checkpoints/mlm/muril-base-cased/` gồm `config.json`
(BertForMaskedLM, vocab 197.285), `model.safetensors` (fp32, có kèm đầu MLM, `train.py` tự bỏ
qua), `tokenizer.json` và `tokenizer_config.json`. Giải nén **ở gốc repo** thì đường dẫn khớp
với các config.

- Đã có thư mục đó thì cell tự bỏ qua, không tải lại.
- Drive báo *quota exceeded*: tải file về máy, rồi *Add Input ▸ Upload* thành một Kaggle Dataset
  và đặt `KAGGLE_ZIP` bên dưới trỏ vào nó (hoặc để trống: cell tự tìm `mlm_muril.zip` trong
  `/kaggle/input`).

In [ ]:
import os, zipfile, glob

MLM_DIR    = "checkpoints/mlm/muril-base-cased"
DRIVE_ID   = "1XgEo2UzdOw8sX4ofd0mmzbOVVhfYnDo7"
KAGGLE_ZIP = ""        # vd "/kaggle/input/mlm-muril/mlm_muril.zip" neu tai qua Kaggle Dataset
ZIP        = "/kaggle/working/mlm_muril.zip"

if os.path.isfile(f"{MLM_DIR}/model.safetensors"):
    print("da co", MLM_DIR, "-> bo qua tai")
else:
    src = KAGGLE_ZIP or (glob.glob("/kaggle/input/**/mlm_muril.zip", recursive=True) or [None])[0]
    if src:
        print("dung zip co san:", src); ZIP = src
    else:
        import gdown
        gdown.download(id=DRIVE_ID, output=ZIP, quiet=False)
    with zipfile.ZipFile(ZIP) as z:
        print("\n".join(f"  {i.file_size / 1e6:9.1f} MB  {i.filename}" for i in z.infolist()))
        z.extractall(".")          # -> ./checkpoints/mlm/muril-base-cased/
    if ZIP.startswith("/kaggle/working/"):
        os.remove(ZIP)             # 1.4 GB it khong can nua; /kaggle/working gioi han ~20 GB
!ls -la {MLM_DIR}

**Kiểm tra checkpoint** (nhanh, không cần GPU): nạp config và tokenizer, so vocab với encoder.
Tokenizer trong zip được lưu bằng transformers 5. Nếu bản trên Kaggle không đọc được, cell sẽ ghi
đè bằng tokenizer của `google/muril-base-cased`. Việc này an toàn vì cùng file vocab và cùng
kiểu giữ chữ hoa/thường; MLM cũng được train bằng chính tokenizer đó.

In [ ]:
import transformers
from transformers import AutoConfig, AutoTokenizer
print("transformers", transformers.__version__)
cfg = AutoConfig.from_pretrained(MLM_DIR)
print("config:", cfg.architectures, "vocab", cfg.vocab_size, "hidden", cfg.hidden_size)
try:
    tok = AutoTokenizer.from_pretrained(MLM_DIR)
    tok.tokenize("Bajetigu thu nin ajji")
except Exception as e:
    print("tokenizer trong zip khong doc duoc:", str(e)[:150], "\n-> thay bang google/muril-base-cased")
    AutoTokenizer.from_pretrained("google/muril-base-cased").save_pretrained(MLM_DIR)
    tok = AutoTokenizer.from_pretrained(MLM_DIR)
enc = AutoTokenizer.from_pretrained("Hate-speech-CNERG/kannada-codemixed-abusive-MuRIL")
print("vocab MLM == vocab cnerg_muril:", tok.get_vocab() == enc.get_vocab())
print("MLM   :", tok.tokenize("Bajetigu thu nin ajji 😡"))
print("cnerg :", enc.tokenize("Bajetigu thu nin ajji 😡"), " (cnerg chuyen chu thuong)")

## 2) Bảng điều khiển

**Mọi thứ chỉnh ở MỘT cell dưới đây.** Cell ngay sau nó chỉ in **kế hoạch** (tên run và đầy đủ
override), không train gì. Xem kế hoạch rồi mới chạy cell train.

Các run được sinh ra:

| run | bật khi | khác mốc ở đâu |
|---|---|---|
| `base` | `RUN_BASE = True` | mốc: encoder thuần, không trộn |
| `mix:<mode>` | mỗi mode trong `MIX_MODES` | + trộn các bảng trong `MIX_SOURCES` |
| `mlm_enc` | `RUN_MLM_ENC = True` | đối chứng: dùng thẳng MuRIL-MLM làm encoder |

`SIDE`, `HYBRID` và `HEAD` được áp cho **mọi** run, kể cả `base` và `mlm_enc`. Như vậy các run
chỉ khác nhau ở phần trộn, và so sánh vẫn công bằng. Chênh lệch dưới ~0,02 macro-F1 là trong
mức nhiễu của lát eval.

In [ ]:
# ============================== BANG DIEU KHIEN ==============================
TASKS = ['a', 'b']                  # 'a' = Hate/Non-Hate, 'b' = 6 nhom doi tuong

# ---- encoder (kiem luon tokenizer) -------------------------------------------------
ENCODER = 'cnerg_muril'             # ten file trong configs/: cnerg_muril | muril | roberta | cnerg_xlmr ...

# ---- embed_mix: tron bang embedding -----------------------------------------------
# Phai DUNG CHUNG VOCAB voi ENCODER (khac vocab -> train.py tu choi va giai thich).
#   voi cnerg_muril / muril:  'google/muril-base-cased', 'checkpoints/mlm/muril-base-cased',
#                             'Hate-speech-CNERG/kannada-codemixed-abusive-MuRIL' (neu ENCODER khac no)
#   voi roberta / cnerg_xlmr: 'xlm-roberta-base', 'Hate-speech-CNERG/deoffxlmr-mono-kannada'
MIX_SOURCES = ['google/muril-base-cased', 'checkpoints/mlm/muril-base-cased']   # [] = khong tron
MIX_MODES   = ['token', 'global']   # moi mode -> 1 run.  token = trong so rieng moi token | global = chung
MIX_LR      = 1e-3                  # lr cua router / trong so tron

# ---- ap cho MOI run -----------------------------------------------------------------
SIDE     = None       # None | 'char' | 'phonetic' | 'char+phonetic'   (vector phu muc tu, gate = 0)
HYBRID   = None       # None | 'tfidf'                                   (ghep TF-IDF truoc head)
HEAD     = 'linear'   # 'linear' | 'mlp'
MLP_DIMS = [512]      # CHI HEAD='mlp'

# ---- huan luyen -------------------------------------------------------------------
EPOCHS   = 6          # cung la do dai lich LR
PATIENCE = 6          # >= EPOCHS: chay tron lich roi giu epoch tot nhat
LR       = 2e-5       # lr cua encoder
BATCH    = 32         # tong tren moi GPU
MAX_LEN  = 96         # token; 128 giam 1/2 so cau bi cat (Religion/Geo-political bi cat nhieu nhat)
LOSS     = 'auto'     # auto (a: ce, b: focal) | ce | wce | focal
SEED     = 42

# ---- run doi chung ----------------------------------------------------------------
RUN_BASE    = True    # encoder khong tron -- moc de so
RUN_MLM_ENC = True    # encoder = checkpoints/mlm/muril-base-cased, khong tron

SUFFIX = None         # None = tu sinh tu cac tham so huan luyen o tren (vd '_e6')
# ===================================================================================

In [ ]:
# ---- KE HOACH: dung bang dieu khien -> danh sach run. Khong train gi o day. ----
import os
from src.utils.config import load_config, run_name

MLM_DIR = 'checkpoints/mlm/muril-base-cased'
fmt = lambda v: str(v).replace(" ", "")              # list -> khong co dau cach cho shell

if SUFFIX is None:                                    # chi ghi nhung gi khac mac dinh
    SUFFIX = f"_e{EPOCHS}"
    for val, dflt, tag in ((LR, 2e-5, 'lr'), (BATCH, 32, 'bs'), (MAX_LEN, 96, 'len'), (PATIENCE, EPOCHS, 'p')):
        if val != dflt:
            SUFFIX += f"_{tag}{val:g}"
    if HEAD == 'mlp':                                 # side/hybrid tu them _se/_hyb, head thi khong
        SUFFIX += "_mlp" + "x".join(map(str, MLP_DIMS))

common = {'training.epochs': EPOCHS, 'training.early_stopping_patience': PATIENCE,
          'training.lr': LR, 'training.batch_size': BATCH, 'data.max_len': MAX_LEN,
          'training.loss': LOSS, 'model.head': HEAD}
if HEAD == 'mlp':
    common['model.mlp_dims'] = MLP_DIMS
if SIDE:
    common['model.side_embedding'] = SIDE
if HYBRID:
    common['model.hybrid'] = HYBRID

PLAN = []   # (nhan, config, overrides)
if RUN_BASE:
    PLAN.append(('base', ENCODER, {}))
if MIX_SOURCES:
    for mode in MIX_MODES:
        PLAN.append((f'mix:{mode}', ENCODER, {'model.embed_mix': MIX_SOURCES,
                                              'model.embed_mix_mode': mode,
                                              'model.embed_mix_lr': MIX_LR}))
if RUN_MLM_ENC:
    if os.path.isfile(f'{MLM_DIR}/config.json'):
        PLAN.append(('mlm_enc', 'muril', {'model.name': MLM_DIR}))
    else:
        print(f"!! RUN_MLM_ENC bo qua: chua co {MLM_DIR} (chay cell tai MLM o muc 1)")
if any(MLM_DIR in str(s) for s in MIX_SOURCES) and not os.path.isfile(f'{MLM_DIR}/config.json'):
    print(f"!! MIX_SOURCES co {MLM_DIR} nhung thu muc chua co -> cac run mix se loi")

JOBS = []
print(f"SUFFIX = {SUFFIX!r}   |   {len(PLAN)} run x {len(TASKS)} task = {len(PLAN) * len(TASKS)} lan train\n")
for label, conf, extra in PLAN:
    sets = {**common, **extra}
    for t in TASKS:
        cfg = load_config(f'configs/{conf}.yaml', [f"{k}={fmt(v)}" for k, v in sets.items()],
                          task=t, seed=SEED, run_suffix=SUFFIX)
        JOBS.append((label, conf, t, sets, run_name(cfg)))
        print(f"  {label:13s} task {t}  ->  {run_name(cfg)}")
print("\noverride chung:", {k: v for k, v in common.items()})

In [ ]:
# ---- TRAIN: chay dung KE HOACH o tren ----
import time
t0 = time.time()
for n, (label, conf, t, sets, name) in enumerate(JOBS, 1):
    args = " ".join(f"{k}={fmt(v)}" for k, v in sets.items())
    print("\n" + "=" * 72)
    print(f"[{n}/{len(JOBS)}]  {label} | {name} | +{(time.time() - t0) / 60:.1f} phut")
    print("=" * 72, flush=True)
    !python train.py --config configs/{conf}.yaml --task {t} --seed {SEED} --set {args} --run_suffix {SUFFIX}
print(f"\nxong {len(JOBS)} run trong {(time.time() - t0) / 60:.1f} phut")

## 3) Kết quả

macro-F1 của đúng các run trong kế hoạch, kèm trọng số trộn và gate mà mỗi run học được (đọc từ
log). Nhớ rằng lát eval nhiễu ±0,02.

In [ ]:
import pandas as pd
names = {name: label for label, _, _, _, name in JOBS}
d = pd.read_csv('results/metrics.csv')
d = d[d.run.isin(names)].assign(nhan=lambda x: x.run.map(names))
display(d[['task', 'nhan', 'run', 'macro_f1', 'accuracy', 'best_epoch']]
        .sort_values(['task', 'macro_f1'], ascending=[True, False]))
for label, conf, t, _, name in JOBS:
    f = f'logs/{t}_{name}.log'
    if os.path.isfile(f):
        lines = [l.split('| INFO | ')[-1].strip() for l in open(f, encoding='utf-8')
                 if 'embed_mix a[' in l or 'side_gate' in l]
        if lines:
            print(f"\n{label} / task {t}:"); print("\n".join("  " + l for l in lines))

In [ ]:
# Blend cac run trong ke hoach (trong so toi uu tren lat eval -> lac quan):
for t in TASKS:
    runs = " ".join(d[d.task == t].run)
    if runs:
        !python evaluate.py --task {t} --runs {runs} --optimize --tag blend_mix

## 4) Tải kết quả về

Nén `results/` và `logs/`. **Không** nén checkpoint: mỗi checkpoint embed_mix nặng thêm khoảng
0,3 GB cho mỗi bảng trộn.

In [ ]:
%cd /kaggle/working/repo
!zip -r -q /kaggle/working/results_embed_mix.zip results logs
!ls -lh /kaggle/working/results_embed_mix.zip